# SecureScan (Phase 1): Asset Discovery + CVE Mapping

This notebook walks through the SecureScan Phase 1 pipeline: **discover** open ports/services on an authorized host, **map** each service to known CVEs (NVD), and produce a **structured report** that is recorded into the same verdict trail as the detectors (`model="scan"`).

**Design highlights**
- **Pluggable engines.** The default `socket` engine is pure-Python (TCP connect scan) so it runs anywhere with no external binary; the optional `nmap` engine adds service/version + CPE detection (available in the Docker image).
- **Authorization guard.** Scans are restricted to loopback/private ranges by default; anything else requires an explicit opt-in (`SCAN_ALLOW_ANY`) — the deliberate switch for a cloud deployment meant to scan whatever environment it is called into.
- **Low-noise by default.** A completed TCP handshake (no raw SYN), optional inter-port delay; no packets are sent to unauthorized hosts.

> Only scan hosts you own or are explicitly authorized to test.

In [ ]:
import sys, os, json
sys.path.insert(0, os.path.join('..', 'backend'))
from securescan import authz, engines, discovery

print('engines available here:', engines.available())
print('default engine       :', engines.get_engine('auto').NAME)
print('authorization posture:', authz.describe())

## 1. The authorization guard
Loopback/private targets are allowed; a public address is refused with a clear reason unless explicitly opted in.

In [ ]:
for t in ['127.0.0.1', '192.168.1.20', '8.8.8.8']:
    print(t, '->', authz.is_authorized(t))

## 2. Discover open ports (socket engine)
We start a throwaway listener on localhost so the scan finds a real open port, then run discovery.

In [ ]:
import socket, threading
srv = socket.socket(); srv.bind(('127.0.0.1', 0)); srv.listen(5)
port = srv.getsockname()[1]
threading.Thread(target=lambda: [srv.accept() for _ in range(3)], daemon=True).start()

report = discovery.scan_and_report('127.0.0.1', ports=[port, 80, 443], engine='socket', with_cves=False)
print(json.dumps(report, indent=2, default=str))

## 3. Map services to CVEs (NVD)
For a service with a product + version (or a CPE from nmap), SecureScan queries the NVD 2.0 API and attaches CVEs with CVSS scores. Results are cached on disk so repeat runs are free and work offline. Set `NVD_API_KEY` for higher rate limits.

Below we enrich a synthetic finding (OpenSSH 7.4) so the notebook is deterministic and does not depend on what is actually listening. In a real scan, `scan_and_report(target, with_cves=True)` does discovery + enrichment in one call.

In [ ]:
from securescan.engines.base import HostScan, PortFinding
hs = HostScan(target='demo', ip='192.168.0.10', up=True, engine='socket',
              ports=[PortFinding(port=22, service='ssh', product='OpenSSH', version='7.4')])
enriched = discovery.enrich_with_cves(hs, max_per_service=5)
print('host max CVSS:', enriched['host_max_cvss'], '| flagged:', enriched['flagged'], '| CVEs:', enriched['cve_total'])
for c in enriched['ports'][0]['cves'][:5]:
    print(f"  {c['cve_id']}  CVSS={c['cvss_score']}  {c['severity']}")

## 4. Record into the platform
`discovery.record_report(report)` writes the scan as a verdict (`model="scan"`, flagged when a high/critical CVE is present, score = host max CVSS) so it appears in the live decisioning trail and dashboard alongside the identity/OT-ICS/compliance decisions.

## 5. Via the API
The same pipeline is exposed at `POST /scan` on the RegMap backend:
```bash
curl -s -X POST http://localhost:2500/scan \
  -H 'Content-Type: application/json' \
  -d '{"target":"127.0.0.1","ports":"1-1024","engine":"auto","with_cves":true}'
```
`GET /scan/engines` reports the usable engines + authorization posture; `GET /scan/authorize?target=...` checks a target without scanning.

## Next phases (see `docs/securescan_roadmap.md`)
- **P2 Hardening assessment** — config/baseline checks per host.
- **P3 CVE → NIST 800-53 control recommendation** — reuse the RegMap embedder.
- **P4 Integration** — feed scan alerts through Decide/Act; show on the dashboard.
- **P5 Continuous monitoring** — scheduled re-scan + drift alerting.